# Add-on B1 — T4/T5 population activity vs decoded MIDD

**Scientific question.** Does static C=48 MIDD generate an Original/Mirror antisymmetric directional signal at T4/T5 population activity level before the artificial optical-flow decoder?

This analysis tests a fixed activity readout in the model. T4/T5 activity and decoded flow are not perception measures; associations do not establish causality. A null result means the prespecified readout did not detect a signal, not that T4/T5 contain no relevant signal.

**Scope and separation from Exp.09.** The existing Exp.09 table contains only model-level, frames 10–49 LOMO-projected summaries for models 000–009; its historical acquisition code and phase are not in this repository. B1 therefore does not claim to reuse that projection or combine its outputs. It uses the existing official-50 phase contract and the FlyVis 1.2.0 network/activity API, with a preregistered subtype-preferred-direction projection. Exp.09 timing is retained. The rendering follows the canonical official-50 C=48 phase analysis (single BoxEye), not Exp.09's centered 8-field stimulus.

**Safety.** This notebook is authored in dry-run mode. No inference, statistics, figures, or output tables are run by default.


## 1. Imports and fixed configuration

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

EXPECTED_FLYVIS_VERSION = "1.2.0"
EXPECTED_FLYVIS_COMMIT = "92b3845cc426dd309a1a0e1b3890156c42e14021"
MODEL_IDS = [f"flow/0000/{i:03d}" for i in range(50)]
MODEL_SHORT_IDS = [f"{i:03d}" for i in range(50)]
SUBTYPES = [f"T4{x}" for x in "abcd"] + [f"T5{x}" for x in "abcd"]
CONDITIONS = ("original", "mirror")
C_LEVEL = 48
DT = 0.02
FPS = 50
N_FRAMES = 50
FADE_SECONDS = 1.0
ANALYSIS_START = 10
ANALYSIS_STOP = 50  # exclusive; analyzed frames are 10–49
IMAGE_SIZE = 382
INNER_DIAMETER = 55
OUTER_DIAMETER = 382
BOXEYE_EXTENT = 15
BOXEYE_KERNEL_SIZE = 13
SEED = 20261006
N_PERMUTATIONS = 20_000
N_BOOTSTRAPS = 10_000

DRY_RUN = False
RUN_INFERENCE = False
RUN_STATISTICS = True
GENERATE_FIGURES = True
SAVE_OPTIONAL_TIMESERIES = False

assert not (DRY_RUN and RUN_INFERENCE), "DRY_RUN and RUN_INFERENCE cannot both be enabled."
assert ANALYSIS_STOP - ANALYSIS_START == 40


## 2. Reproducibility metadata and repository paths

In [ ]:
ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "data/precomputed/fig04/phase_stimulus_manifest.csv").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from within the flyvis-midd checkout.")

PHASE_MANIFEST = ROOT / "data/precomputed/fig04/phase_stimulus_manifest.csv"
PHASE_PAIRS = ROOT / "data/precomputed/fig04/official50_phase_pairs.csv"
PHASE_CONFIG = ROOT / "data/precomputed/fig04/phase_config.json"
EXP09_ACTIVITY = ROOT / "data/precomputed/fig06/exp09_activity_by_model_luminance.csv"
EXP09_MANIFEST = ROOT / "data/precomputed/fig06/exp09_manifest.json"
EXP09_EVIDENCE = ROOT / "data/precomputed/fig06/exp09_evidence_metadata.json"
B0_INVENTORY = ROOT / "data/precomputed/supplementary/biological_constraints/B/activity_inventory.csv"
A1_MEMBERSHIP = ROOT / "data/precomputed/supplementary/biological_constraints/A1/t4c_cluster_membership.csv"
OUTPUT_DIR = ROOT / "data/precomputed/supplementary/biological_constraints/B1"
CACHE_DIR = OUTPUT_DIR / "cache"
ACTIVITY_CSV = OUTPUT_DIR / "t4t5_activity_model_phase.csv"
TIMESERIES_NPZ = OUTPUT_DIR / "t4t5_activity_timeseries.npz"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

try:
    import flyvis
    flyvis_version = getattr(flyvis, "__version__", "unknown")
    flyvis_module = str(Path(flyvis.__file__).resolve())
    pretrained_root = Path(flyvis.results_dir)
except Exception as exc:
    flyvis = None
    flyvis_version = "unavailable"
    flyvis_module = f"unavailable: {type(exc).__name__}: {exc}"
    pretrained_root = None

try:
    import torch
    torch_version = torch.__version__
    device = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
except Exception:
    torch = None
    torch_version = "unavailable"
    device = "unavailable"

try:
    import subprocess
    git_commit = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=ROOT, check=True,
        capture_output=True, text=True,
    ).stdout.strip()
except Exception:
    git_commit = "unavailable"

print({
    "python": platform.python_version(),
    "FlyVis": flyvis_version,
    "FlyVis_module": flyvis_module,
    "PyTorch": torch_version,
    "device": device,
    "repository_commit": git_commit,
    "expected_FlyVis_commit": EXPECTED_FLYVIS_COMMIT,
    "seed": SEED,
    "models": "flow/0000/000–049 in canonical ordering",
    "stimulus": "3MIDD C=48 Original/Mirror, all canonical phases",
    "timing": {"fps": FPS, "dt": DT, "frames": N_FRAMES, "fade_in_seconds": FADE_SECONDS, "analysis_frames": "10–49"},
})


## 3. Load and verify existing B0 / Exp.09 inventory

In [ ]:
assert EXP09_ACTIVITY.is_file() and EXP09_MANIFEST.is_file() and EXP09_EVIDENCE.is_file()
exp09 = pd.read_csv(EXP09_ACTIVITY)
exp09_manifest = json.loads(EXP09_MANIFEST.read_text())
exp09_evidence = json.loads(EXP09_EVIDENCE.read_text())
b0 = pd.read_csv(B0_INVENTORY) if B0_INVENTORY.is_file() else None

assert exp09_manifest["experiment"] == "Experiment09_MIDDvsT4T5"
assert exp09_manifest["temporal"]["dt_sec"] == DT
assert exp09_manifest["temporal"]["fade_in_state_sec"] == FADE_SECONDS
assert exp09_manifest["temporal"]["n_frames"] == N_FRAMES
assert exp09_manifest["temporal"]["analysis_frame_start"] == ANALYSIS_START
assert exp09_manifest["temporal"]["analysis_frame_stop"] == ANALYSIS_STOP
assert exp09_evidence["analysis_frames"] == list(range(ANALYSIS_START, ANALYSIS_STOP))
assert exp09_manifest["model_ids"] == [f"{i:03d}" for i in range(10)]
assert set(SUBTYPES).issubset(set(exp09.cell_type))
assert not exp09.duplicated(["model_id", "cell_type", "c"]).any()

exp09_c48 = exp09.loc[exp09.c.eq(C_LEVEL) & exp09.cell_type.isin(SUBTYPES)]
assert len(exp09_c48) == 10 * len(SUBTYPES)
assert exp09_c48[["e_original_projected", "e_mirror_projected"]].notna().all().all()
print({
    "Exp09_rows_at_C48_T4T5": len(exp09_c48),
    "Exp09_models": sorted(exp09_c48.model_id.unique().tolist()),
    "existing_projection": "LOMO common preferred-direction tangent; saved summaries only",
    "Exp09_phase_known": False,
    "Exp09_acquisition_code_in_repository": False,
    "B0_inventory_loaded": b0 is not None,
})


## 4. Canonical C=48 phases and stimulus contract

In [ ]:
phase_manifest = pd.read_csv(PHASE_MANIFEST)
phase_config = json.loads(PHASE_CONFIG.read_text())
phase_pairs_existing = pd.read_csv(PHASE_PAIRS)
c48_manifest = phase_manifest.loc[phase_manifest.group.eq("3MIDD_048")].copy()
phase_rows = c48_manifest.loc[c48_manifest.condition.str.endswith("__original")].sort_values("phase_fraction")
PHASES = phase_rows[["phase_fraction", "phase_deg"]].drop_duplicates().reset_index(drop=True)

expected_phase_deg = np.arange(12, dtype=float) * 15.0 / 12.0
assert len(PHASES) == 12
assert np.allclose(PHASES.phase_deg.to_numpy(), expected_phase_deg, atol=1e-12)
assert np.allclose(PHASES.phase_fraction.to_numpy(), np.arange(12) / 12, atol=1e-12)
assert set(c48_manifest.condition.str.split("__").str[-1]) == {"original", "mirror", "negative"}
assert len(c48_manifest) == 12 * 3
assert set(phase_pairs_existing.group) == {"3MIDD_048"}
assert phase_pairs_existing.groupby("model_id").phase_index.nunique().eq(12).all()
assert phase_config["dt"] == DT and phase_config["fade"] == FADE_SECONDS
assert phase_config["frames"] == N_FRAMES
assert phase_config["window"] == [ANALYSIS_START, ANALYSIS_STOP]
assert phase_config["image_size"] == IMAGE_SIZE
assert phase_config["diameters"] == [INNER_DIAMETER, OUTER_DIAMETER]
assert phase_config["BoxEye"] == [BOXEYE_EXTENT, BOXEYE_KERNEL_SIZE]
assert phase_config["phase_period"].startswith("full repeated color unit;15deg")

print("Canonical phase values (degrees):", PHASES.phase_deg.tolist())
print("C=48 source contracts:", c48_manifest[["phase_deg", "condition", "image_sha256", "rendered_sha256"]].to_dict("records")[:2])
print("Canonical rendering: single BoxEye, extent=15, kernel=13; 382 px image; inner/outer diameter 55/382.")


## 5. Official T4/T5 direction mapping and fixed readout

The FlyVis 1.2.0 official moving-bar response source defines the known preferred directions in the order a–d as π, 0, π/2, and 3π/2 radians for both T4 (ON pathway) and T5 (OFF pathway). This notebook uses those fixed model-coordinate directions; it does not estimate weights from MIDD or flip signs after seeing results. The official documentation describes the connectome v coordinate as the 60-degree principal axis.

For each subtype and receptor position, the scalar directional weight is the cosine between its fixed preferred-direction vector and the local clockwise tangent. Lattice coordinates are mapped to Cartesian coordinates using the documented hex basis (u along x; v at +60° in y-up coordinates). The primary readout is the equal-weight mean of the eight subtype projections; T4-only, T5-only, and individual subtypes are secondary. This is a prespecified model activity readout, not a biological vector-flow measurement.

The historical Exp.09 LOMO projection implementation is unavailable in this repository, so this fixed projection is intentionally reported as a distinct B1 readout. The source mapping is checked at runtime against the requested eight cell types and node coordinates.


In [ ]:
# Official FlyVis moving_bar_responses.angular_distance_to_known:
# T4a-d and T5a-d known directions = [pi, 0, pi/2, 3pi/2], respectively.
# Source: https://turagalab.github.io/flyvis/reference/moving_stimulus_responses/
PREFERRED_DIRECTION_RAD = {
    "T4a": math.pi, "T4b": 0.0, "T4c": math.pi / 2, "T4d": 3 * math.pi / 2,
    "T5a": math.pi, "T5b": 0.0, "T5c": math.pi / 2, "T5d": 3 * math.pi / 2,
}
assert set(PREFERRED_DIRECTION_RAD) == set(SUBTYPES)
READOUT_NAME = "equal-subtype T4+T5 directional activity projected onto local clockwise tangent"
BASELINE_POLICY = "no additional baseline subtraction; same fixed fade-in and frames 10–49 for both conditions"


## 6. Estimate the workload and verify pretrained model paths (dry run)

In [ ]:
N_MODELS = len(MODEL_IDS)
N_PHASES = len(PHASES)
N_CONDITIONS = len(CONDITIONS)
N_SIMULATIONS = N_MODELS * N_PHASES * N_CONDITIONS
N_LONG_ROWS = N_MODELS * N_PHASES * N_CONDITIONS * len(SUBTYPES)
ESTIMATED_SCALAR_TIMESERIES_BYTES = N_MODELS * N_PHASES * N_CONDITIONS * len(SUBTYPES) * N_FRAMES * np.dtype("float32").itemsize
model_paths = [] if pretrained_root is None else [pretrained_root / model_id for model_id in MODEL_IDS]
missing_model_paths = [str(p) for p in model_paths if not p.exists()]

assert MODEL_IDS == [f"flow/0000/{i:03d}" for i in range(50)]
assert N_SIMULATIONS == 1200
assert N_LONG_ROWS == 9600
print({
    "planned_models": N_MODELS,
    "planned_phases": N_PHASES,
    "planned_conditions_per_phase": N_CONDITIONS,
    "network_simulations_if_enabled": N_SIMULATIONS,
    "subtype_condition_summary_rows": N_LONG_ROWS,
    "optional_scalar_time_series_size_MiB": round(ESTIMATED_SCALAR_TIMESERIES_BYTES / 2**20, 2),
    "pretrained_root": str(pretrained_root) if pretrained_root else "FlyVis unavailable",
    "existing_model_paths_found": len(model_paths) - len(missing_model_paths),
    "missing_model_paths": missing_model_paths[:5],
    "planned_output": str(ACTIVITY_CSV),
    "optional_output": str(TIMESERIES_NPZ) if SAVE_OPTIONAL_TIMESERIES else "disabled",
    "DRY_RUN": DRY_RUN,
    "RUN_INFERENCE": RUN_INFERENCE,
})


## 7. Fixed stimulus rendering and activity extraction functions

In [ ]:
def make_boxeye_movie(phase_deg, condition):
    """Render the canonical 3MIDD C=48 image, repeat it for 50 frames, then BoxEye-render."""
    if condition not in CONDITIONS:
        raise ValueError(f"Unsupported condition: {condition}")
    from flyvis_midd.stimuli import build_3midd_sequences, render_sector_image
    from flyvis.datasets.rendering import BoxEye

    seq = build_3midd_sequences(C_LEVEL)[condition]
    image, _ = render_sector_image(
        seq, image_size=IMAGE_SIZE, inner_diameter=INNER_DIAMETER,
        outer_diameter=OUTER_DIAMETER, phase_deg=float(phase_deg),
    )
    movie = np.repeat(image[None, None, :, :], N_FRAMES, axis=1).astype(np.float32) / 255.0
    rendered = BoxEye(extent=BOXEYE_EXTENT, kernel_size=BOXEYE_KERNEL_SIZE)(
        torch.as_tensor(movie)
    )
    if tuple(rendered.shape) != (1, N_FRAMES, 1, 721):
        raise AssertionError(f"Unexpected BoxEye movie shape: {tuple(rendered.shape)}")
    return rendered


def lattice_xy_from_node_table(nodes):
    """Return fixed y-up Cartesian coordinates from official node u/v lattice coordinates."""
    if not {"index", "type", "u", "v"}.issubset(nodes.columns):
        raise ValueError(f"Unexpected FlyVis connectome node columns: {list(nodes.columns)}")
    # FlyVis hex-grid convention: u is the x principal axis and v is 60 degrees from u.
    x = nodes["u"].to_numpy(float) + 0.5 * nodes["v"].to_numpy(float)
    y = (math.sqrt(3) / 2.0) * nodes["v"].to_numpy(float)
    return x, y


def subtype_projection_weights(node_rows, subtype):
    """Cosine projection to subtype preferred direction vs. local CW tangent; no fitting."""
    x, y = lattice_xy_from_node_table(node_rows)
    radius = np.hypot(x, y)
    theta_radial = np.arctan2(y, x)
    theta_cw_tangent = theta_radial - math.pi / 2
    weights = np.cos(PREFERRED_DIRECTION_RAD[subtype] - theta_cw_tangent)
    weights[radius == 0] = np.nan  # tangent is undefined at the visual-field origin
    return weights


def extract_condition_activity(network, movie_input, model_id, phase_index, phase_deg, condition):
    """Run one guarded future inference and return compact subtype readouts/traces."""
    import torch
    network.eval()
    if network.training:
        raise RuntimeError("Expected the loaded FlyVis network to be in evaluation mode.")
    # FlyVis 1.2.0 network.simulate enters its official simulation context,
    # which temporarily disables parameter gradients and restores them on exit.
    # Parameters may therefore have requires_grad=True before simulate is called.
    initial_state = network.fade_in_state(FADE_SECONDS, DT, movie_input[:, 0])
    with torch.no_grad():
        activity = network.simulate(movie_input, DT, initial_state=initial_state)
    activity = activity.detach().cpu()
    node_df = network.connectome.nodes.to_df().reset_index(drop=True)
    if len(node_df) != activity.shape[-1]:
        raise AssertionError("Connectome node table and activity vector differ in size.")
    frames = activity[0, ANALYSIS_START:ANALYSIS_STOP].numpy()
    records, traces = [], {}
    subtype_summaries = {}
    for subtype in SUBTYPES:
        rows = node_df.loc[node_df["type"].eq(subtype)].copy()
        if rows.empty:
            raise AssertionError(f"Missing requested cell subtype {subtype}")
        indices = rows["index"].to_numpy(dtype=int)
        if np.any(indices < 0) or np.any(indices >= activity.shape[-1]):
            raise AssertionError(f"Invalid node indices for {subtype}")
        weights = subtype_projection_weights(rows, subtype)
        valid = np.isfinite(weights)
        if not valid.any():
            raise AssertionError(f"No noncentral cells available for {subtype}")
        indices, weights = indices[valid], weights[valid]
        trace = np.mean(frames[:, indices] * weights[None, :], axis=1)
        if not np.isfinite(trace).all():
            raise FloatingPointError(f"Nonfinite activity for {model_id}/{phase_index}/{condition}/{subtype}")
        traces[subtype] = trace.astype(np.float32)
        subtype_summaries[subtype] = float(trace.mean())
        records.append({
            "model_id": model_id, "phase_index": int(phase_index),
            "phase_fraction": float(PHASES.loc[phase_index, "phase_fraction"]),
            "phase_deg": float(phase_deg), "condition": condition,
            "cell_group": subtype, "response": float(trace.mean()),
            "analysis_start_frame": ANALYSIS_START,
            "analysis_end_frame_exclusive": ANALYSIS_STOP,
            "n_frames": len(trace), "readout": READOUT_NAME,
            "baseline_policy": BASELINE_POLICY,
        })
    del activity
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return records, traces, subtype_summaries


## 8. Cache contract and guarded inference loop

In [ ]:
def cache_path(model_short_id, phase_index):
    return CACHE_DIR / f"model_{model_short_id}_phase_{phase_index:02d}.json"

def cache_signature(model_id, phase_index, phase_deg):
    return {
        "model_id": model_id,
        "phase_index": int(phase_index),
        "phase_deg": float(phase_deg),
        "conditions": list(CONDITIONS),
        "subtypes": list(SUBTYPES),
        "dt": DT,
        "fade_seconds": FADE_SECONDS,
        "n_frames": N_FRAMES,
        "analysis_window": [ANALYSIS_START, ANALYSIS_STOP],
        "stimulus": "3MIDD_C48_single_BoxEye",
        "readout": READOUT_NAME,
        "preferred_direction_rad": PREFERRED_DIRECTION_RAD,
        "FlyVis_version": EXPECTED_FLYVIS_VERSION,
        "FlyVis_commit": EXPECTED_FLYVIS_COMMIT,
    }

def load_valid_cache(path, signature):
    if not path.exists():
        return None
    try:
        payload = json.loads(path.read_text())
        if payload.get("signature") != signature:
            raise ValueError("cache signature mismatch")
        rows = pd.DataFrame(payload["rows"])
        expected = {(c, s) for c in CONDITIONS for s in SUBTYPES}
        got = set(zip(rows.condition, rows.cell_group))
        if len(rows) != len(expected) or got != expected:
            raise ValueError("partial or duplicate model-phase cache")
        if not np.isfinite(rows.response.to_numpy(float)).all():
            raise ValueError("nonfinite cached activity")
        return payload
    except Exception as exc:
        raise RuntimeError(f"Partial/corrupt cache must be moved aside and audited: {path}: {exc}") from exc

def atomic_json_write(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True))
    temporary.replace(path)

def run_inference():
    if DRY_RUN or not RUN_INFERENCE:
        print("SKIPPED: DRY_RUN=True or RUN_INFERENCE=False; no FlyVis inference was run.")
        return None
    if flyvis is None or flyvis_version != EXPECTED_FLYVIS_VERSION:
        raise RuntimeError(f"Expected FlyVis {EXPECTED_FLYVIS_VERSION}; found {flyvis_version}.")
    if missing_model_paths:
        raise FileNotFoundError(f"Missing official pretrained models, first entries: {missing_model_paths[:5]}")
    import torch
    from flyvis import NetworkView

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    all_rows = []
    all_traces = {}
    for model_id, short_id in zip(MODEL_IDS, MODEL_SHORT_IDS):
        network_view = NetworkView(pretrained_root / model_id)
        network = network_view.init_network()
        network.eval()
        for phase_index, phase_deg in enumerate(PHASES.phase_deg.tolist()):
            signature = cache_signature(model_id, phase_index, phase_deg)
            path = cache_path(short_id, phase_index)
            cached = load_valid_cache(path, signature)
            if cached is not None:
                all_rows.extend(cached["rows"])
                if SAVE_OPTIONAL_TIMESERIES:
                    all_traces.update(cached.get("traces", {}))
                continue
            pair_rows, pair_traces = [], {}
            for condition in CONDITIONS:
                movie_input = make_boxeye_movie(phase_deg, condition)
                rows, traces, _ = extract_condition_activity(
                    network, movie_input, model_id, phase_index, phase_deg, condition
                )
                pair_rows.extend(rows)
                if SAVE_OPTIONAL_TIMESERIES:
                    for subtype, trace in traces.items():
                        pair_traces[f"{model_id}|{phase_index}|{condition}|{subtype}"] = trace.tolist()
                del movie_input
            payload = {"signature": signature, "rows": pair_rows, "traces": pair_traces}
            atomic_json_write(path, payload)
            all_rows.extend(pair_rows)
            if SAVE_OPTIONAL_TIMESERIES:
                all_traces.update(pair_traces)
            # Keep one model resident across its 12 phase pairs; only one network is loaded at a time.
        del network, network_view
    if len(all_rows) != N_LONG_ROWS:
        raise AssertionError(f"Expected {N_LONG_ROWS * N_PHASES} activity rows, got {len(all_rows)}")
    activity_long = pd.DataFrame(all_rows)
    assert not activity_long.duplicated(["model_id", "phase_index", "condition", "cell_group"]).any()
    assert set(activity_long.model_id) == set(MODEL_IDS)
    assert set(activity_long.phase_index) == set(range(N_PHASES))
    assert set(activity_long.condition) == set(CONDITIONS)
    assert set(activity_long.cell_group) == set(SUBTYPES)
    if not np.isfinite(activity_long.response.to_numpy(float)).all():
        raise FloatingPointError("Nonfinite activity values.")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    temporary_csv = ACTIVITY_CSV.with_suffix(".csv.tmp")
    activity_long.to_csv(temporary_csv, index=False)
    temporary_csv.replace(ACTIVITY_CSV)
    if SAVE_OPTIONAL_TIMESERIES:
        np.savez_compressed(TIMESERIES_NPZ, **{k: np.asarray(v, dtype=np.float32) for k, v in all_traces.items()})
    return activity_long

if DRY_RUN or not RUN_INFERENCE:
    print("Inference cell is guarded and skipped.")
else:
    activity_long = run_inference()


## 9. Load and validate completed activity (only after an explicit run)

In [ ]:
activity_long = None
if ACTIVITY_CSV.is_file():
    activity_long = pd.read_csv(ACTIVITY_CSV)
    required = {
        "model_id", "phase_index", "phase_fraction", "phase_deg", "condition",
        "cell_group", "response", "analysis_start_frame", "analysis_end_frame_exclusive",
    }
    assert required.issubset(activity_long.columns)
    assert set(activity_long.model_id) == set(MODEL_IDS)
    assert set(activity_long.phase_index) == set(range(N_PHASES))
    assert set(activity_long.condition) == set(CONDITIONS)
    assert set(activity_long.cell_group) == set(SUBTYPES)
    assert not activity_long.duplicated(["model_id", "phase_index", "condition", "cell_group"]).any()
    assert (activity_long.analysis_start_frame == ANALYSIS_START).all()
    assert (activity_long.analysis_end_frame_exclusive == ANALYSIS_STOP).all()
    assert np.isfinite(activity_long.response.to_numpy(float)).all()
    print("Completed activity table loaded:", activity_long.shape)
else:
    print("SKIPPED: activity CSV is absent. This is expected before B1 inference.")


## 10. Derive model × phase antisymmetric activity and join decoded flow

In [ ]:
activity_phase = decoded_phase = paired_phase = None
activity_model = flow_model = model_phase_means = None
if activity_long is not None:
    wide = activity_long.pivot(
        index=["model_id", "phase_index", "phase_fraction", "phase_deg", "cell_group"],
        columns="condition", values="response",
    ).reset_index()
    assert set(wide.columns).issuperset({"original", "mirror"})
    assert wide[["original", "mirror"]].notna().all().all()
    wide["A_activity"] = (wide.original - wide.mirror) / 2.0

    def aggregate_groups(table):
        sub = table.pivot(
            index=["model_id", "phase_index", "phase_fraction", "phase_deg", "condition"],
            columns="cell_group", values="response",
        ).reset_index()
        sub["T4"] = sub[[f"T4{x}" for x in "abcd"]].mean(axis=1)
        sub["T5"] = sub[[f"T5{x}" for x in "abcd"]].mean(axis=1)
        sub["T4T5"] = sub[SUBTYPES].mean(axis=1)
        long = sub.melt(
            id_vars=["model_id", "phase_index", "phase_fraction", "phase_deg", "condition"],
            value_vars=["T4", "T5", "T4T5"],
            var_name="cell_group", value_name="response",
        )
        return long

    # Preserve every subtype's paired condition contrast for the exploratory table.
    subtype_phase = wide[
        ["model_id", "phase_index", "phase_fraction", "phase_deg", "cell_group", "A_activity"]
    ].copy()
    group_long = aggregate_groups(activity_long)
    group_wide = group_long.pivot(
        index=["model_id", "phase_index", "phase_fraction", "phase_deg", "cell_group"],
        columns="condition", values="response",
    ).reset_index()
    group_wide["A_activity"] = (group_wide.original - group_wide.mirror) / 2.0
    group_phase = group_wide[
        ["model_id", "phase_index", "phase_fraction", "phase_deg", "cell_group", "A_activity"]
    ].copy()
    activity_phase = pd.concat([subtype_phase, group_phase], ignore_index=True)
    assert not activity_phase.duplicated(["model_id", "phase_index", "cell_group"]).any()
    assert np.isfinite(activity_phase.A_activity.to_numpy(float)).all()

    decoded = pd.read_csv(PHASE_PAIRS)
    decoded = decoded.loc[decoded.group.eq("3MIDD_048")].copy()
    decoded["model_id"] = decoded.model_id.map(lambda i: f"flow/0000/{int(i):03d}")
    phase_lookup = PHASES.reset_index(names="phase_index")
    decoded = decoded.merge(phase_lookup, on="phase_index", how="left", validate="many_to_one")
    assert decoded.phase_deg.notna().all()
    assert not decoded.duplicated(["model_id", "phase_index"]).any()
    assert len(decoded) == 50 * 12
    primary_activity = activity_phase.loc[activity_phase.cell_group.eq("T4T5")]
    paired_phase = primary_activity.merge(
        decoded[["model_id", "phase_index", "A"]].rename(columns={"A":"A_flow"}),
        on=["model_id", "phase_index"], how="inner", validate="one_to_one",
    )
    assert len(paired_phase) == 50 * 12
    activity_model = primary_activity.groupby("model_id", as_index=False).A_activity.mean()
    flow_model = decoded.groupby("model_id", as_index=False).A.mean().rename(columns={"A":"A_flow"})
    model_phase_means = activity_model.merge(flow_model, on="model_id", validate="one_to_one")
    assert len(model_phase_means) == 50
    print({"paired_model_phase_rows": len(paired_phase), "model_level_units": len(model_phase_means),
           "subtype_phase_rows": len(subtype_phase), "group_phase_rows": len(group_phase)})
else:
    print("SKIPPED: no completed activity table; no activity contrasts or joins computed.")

## 11. Prespecified primary statistics (model is the unit; phase is repeated)

In [ ]:
from scipy.stats import binomtest, rankdata, spearmanr

def _require_model_vectors(x, y=None):
    x = np.asarray(x, dtype=float)
    if x.ndim != 1 or not np.isfinite(x).all():
        raise ValueError("Expected a finite one-dimensional model-level vector.")
    if y is not None:
        y = np.asarray(y, dtype=float)
        if y.shape != x.shape or y.ndim != 1 or not np.isfinite(y).all():
            raise ValueError("Expected a paired finite model-level vector.")
    return (x,) if y is None else (x, y)

def bootstrap_mean_ci(values, n_boot=N_BOOTSTRAPS, seed=SEED):
    (x,) = _require_model_vectors(values)
    rng = np.random.default_rng(seed)
    draws = rng.choice(x, size=(n_boot, len(x)), replace=True).mean(axis=1)
    return np.quantile(draws, [0.025, 0.975])

def sign_flip_test(values, n_perm=N_PERMUTATIONS, seed=SEED):
    (x,) = _require_model_vectors(values)
    observed = abs(float(x.mean()))
    rng = np.random.default_rng(seed)
    exceed = 0
    for start in range(0, n_perm, 500):
        n = min(500, n_perm - start)
        signs = rng.choice(np.array([-1.0, 1.0]), size=(n, len(x)))
        exceed += int(np.sum(np.abs((signs * x).mean(axis=1)) >= observed - 1e-15))
    return (exceed + 1) / (n_perm + 1)

def spearman_permutation(x, y, n_perm=N_PERMUTATIONS, seed=SEED):
    x, y = _require_model_vectors(x, y)
    observed = float(spearmanr(x, y).statistic)
    rx, ry = rankdata(x), rankdata(y)
    rx = rx - rx.mean()
    ry = ry - ry.mean()
    denom = np.sqrt(np.sum(rx ** 2) * np.sum(ry ** 2))
    rng = np.random.default_rng(seed)
    exceed = 0
    for start in range(0, n_perm, 1000):
        n = min(1000, n_perm - start)
        order = np.argsort(rng.random((n, len(y))), axis=1)
        permuted = ry[order]
        stats = permuted @ rx / denom
        exceed += int(np.sum(np.abs(stats) >= abs(observed) - 1e-15))
    return observed, (exceed + 1) / (n_perm + 1)

def bootstrap_spearman_ci(x, y, n_boot=N_BOOTSTRAPS, seed=SEED):
    x, y = _require_model_vectors(x, y)
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(x), size=(n_boot, len(x)))
    xb, yb = x[indices], y[indices]
    rxb = rankdata(xb, axis=1)
    ryb = rankdata(yb, axis=1)
    rxb -= rxb.mean(axis=1, keepdims=True)
    ryb -= ryb.mean(axis=1, keepdims=True)
    numerator = np.sum(rxb * ryb, axis=1)
    denominator = np.sqrt(np.sum(rxb ** 2, axis=1) * np.sum(ryb ** 2, axis=1))
    rho = numerator / denominator
    rho = rho[np.isfinite(rho)]
    if len(rho) < n_boot * 0.95:
        raise RuntimeError("Too many degenerate model-level bootstrap samples for Spearman CI.")
    return np.quantile(rho, [0.025, 0.975])

def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    out = np.empty_like(p)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (len(p) - rank) * p[idx])
        out[idx] = min(1.0, running)
    return out

assert activity_model is not None and RUN_STATISTICS
assert not RUN_INFERENCE, "Statistics-only run must never enable FlyVis inference."
assert len(model_phase_means) == 50 and model_phase_means.model_id.nunique() == 50

x = model_phase_means.A_activity.to_numpy(float)
y = model_phase_means.A_flow.to_numpy(float)
p1_ci = bootstrap_mean_ci(x, seed=SEED)
p1 = sign_flip_test(x, seed=SEED)
p2_rho, p2 = spearman_permutation(x, y, seed=SEED + 1)
p2_ci = bootstrap_spearman_ci(x, y, seed=SEED + 2)
keep = (x != 0) & (y != 0)
n_zero_excluded = int((~keep).sum())
n_agree = int(np.sum(np.sign(x[keep]) == np.sign(y[keep])))
n_sign = int(keep.sum())
p3_prop = n_agree / n_sign
p3_test = binomtest(n_agree, n_sign, p=0.5, alternative="two-sided")
p3_ci = p3_test.proportion_ci(confidence_level=0.95, method="exact")
primary_p = np.array([p1, p2, float(p3_test.pvalue)])
primary_holm = holm_adjust(primary_p)
primary_statistics = pd.DataFrame([
    {"family":"primary", "endpoint":"P1_mean_A_activity", "n":len(x),
     "statistic":"two-sided model-level sign-flip", "effect_size":float(x.mean() / x.std(ddof=1)),
     "estimate":float(x.mean()), "ci_low":float(p1_ci[0]), "ci_high":float(p1_ci[1]),
     "raw_p":float(p1), "holm_p":float(primary_holm[0]), "n_excluded":0,
     "note":"Mean and percentile bootstrap 95% CI; Cohen dz; 20,000 sign flips."},
    {"family":"primary", "endpoint":"P2_spearman_A_activity_vs_A_flow", "n":len(x),
     "statistic":"two-sided model-level permutation Spearman", "effect_size":float(p2_rho),
     "estimate":float(p2_rho), "ci_low":float(p2_ci[0]), "ci_high":float(p2_ci[1]),
     "raw_p":float(p2), "holm_p":float(primary_holm[1]), "n_excluded":0,
     "note":"Paired percentile bootstrap 95% CI; 20,000 model-label permutations."},
    {"family":"primary", "endpoint":"P3_sign_agreement", "n":n_sign,
     "statistic":"two-sided exact binomial against 0.5", "effect_size":float(p3_prop),
     "estimate":float(p3_prop), "ci_low":float(p3_ci.low), "ci_high":float(p3_ci.high),
     "raw_p":float(p3_test.pvalue), "holm_p":float(primary_holm[2]), "n_excluded":n_zero_excluded,
     "note":f"{n_agree}/{n_sign} nonzero pairs agree; exact-zero pairs excluded."},
])
primary_statistics.to_csv(OUTPUT_DIR / "t4t5_stats.csv", index=False, float_format="%.17g")
print(primary_statistics.to_string(index=False))

activity_model = model_phase_means.copy()
model_summary = activity_model.rename(columns={"A_activity":"phase_mean_A_activity","A_flow":"phase_mean_A_flow"})
primary_phase = activity_phase.loc[activity_phase.cell_group.eq("T4T5")].copy()
phase0 = primary_phase.loc[primary_phase.phase_index.eq(0), ["model_id","A_activity"]].rename(columns={"A_activity":"phase0_A_activity"})
phase_shape = primary_phase.sort_values("phase_index").groupby("model_id").A_activity
phase_sd = phase_shape.std(ddof=1).rename("activity_phase_SD")
sign_changes = phase_shape.apply(
    lambda s: (lambda signs: bool(len(signs) > 1 and np.any(signs[1:] != signs[:-1])))(np.sign(s.to_numpy())[np.sign(s.to_numpy()) != 0])
).rename("activity_sign_change")
t4t5_means = activity_phase.loc[activity_phase.cell_group.isin(["T4","T5"])].groupby(
    ["model_id","cell_group"], as_index=False
).A_activity.mean().pivot(index="model_id",columns="cell_group",values="A_activity").rename(
    columns={"T4":"T4_phase_mean_A_activity","T5":"T5_phase_mean_A_activity"}
).reset_index()
model_summary = model_summary.merge(phase0,on="model_id",validate="one_to_one")
model_summary = model_summary.merge(phase_sd,on="model_id",validate="one_to_one")
model_summary = model_summary.merge(sign_changes,on="model_id",validate="one_to_one")
model_summary = model_summary.merge(t4t5_means,on="model_id",validate="one_to_one")
model_summary["activity_sign"] = np.sign(model_summary.phase_mean_A_activity).astype(int)
model_summary["flow_sign"] = np.sign(model_summary.phase_mean_A_flow).astype(int)
model_summary["sign_agreement"] = np.where(
    (model_summary.activity_sign != 0) & (model_summary.flow_sign != 0),
    model_summary.activity_sign == model_summary.flow_sign, np.nan
)
model_summary = model_summary.sort_values("model_id").reset_index(drop=True)
assert len(model_summary) == 50 and model_summary.model_id.is_unique
model_summary.to_csv(OUTPUT_DIR / "t4t5_model_summary.csv",index=False,float_format="%.17g")

secondary_rows = []
secondary_means = {}
for group, offset in [("T4", 11), ("T5", 21)]:
    g = activity_phase.loc[activity_phase.cell_group.eq(group)].copy()
    gm = g.groupby("model_id",as_index=False).A_activity.mean().merge(
        flow_model,on="model_id",validate="one_to_one")
    gx, gy = gm.A_activity.to_numpy(float), gm.A_flow.to_numpy(float)
    ci = bootstrap_mean_ci(gx, seed=SEED+offset)
    pmean = sign_flip_test(gx, seed=SEED+offset)
    rho, prho = spearman_permutation(gx,gy,seed=SEED+offset+1)
    rci = bootstrap_spearman_ci(gx,gy,seed=SEED+offset+2)
    secondary_means[group] = gm.set_index("model_id").A_activity
    secondary_rows.extend([
        {"family":"secondary", "endpoint":f"{group}_mean_A_activity", "n":50,
         "statistic":"two-sided model-level sign-flip", "effect_size":float(gx.mean()/gx.std(ddof=1)),
         "estimate":float(gx.mean()),"ci_low":float(ci[0]),"ci_high":float(ci[1]),
         "raw_p":float(pmean),"n_excluded":0,"note":"Exploratory; percentile bootstrap 95% CI."},
        {"family":"secondary", "endpoint":f"{group}_spearman_vs_A_flow", "n":50,
         "statistic":"two-sided model-level permutation Spearman","effect_size":float(rho),
         "estimate":float(rho),"ci_low":float(rci[0]),"ci_high":float(rci[1]),
         "raw_p":float(prho),"n_excluded":0,"note":"Exploratory; paired percentile bootstrap 95% CI."},
    ])
t4v = secondary_means["T4"].reindex(model_summary.model_id).to_numpy(float)
t5v = secondary_means["T5"].reindex(model_summary.model_id).to_numpy(float)
diff=t4v-t5v
diffci=bootstrap_mean_ci(diff,seed=SEED+31)
pdiff=sign_flip_test(diff,seed=SEED+31)
secondary_rows.append({"family":"secondary","endpoint":"paired_T4_minus_T5_mean_A","n":50,
    "statistic":"two-sided paired model-level sign-flip","effect_size":float(diff.mean()/diff.std(ddof=1)),
    "estimate":float(diff.mean()),"ci_low":float(diffci[0]),"ci_high":float(diffci[1]),
    "raw_p":float(pdiff),"n_excluded":0,"note":"Exploratory paired difference; network is the unit."})
phase0_x=phase0.set_index("model_id").reindex(model_summary.model_id).phase0_A_activity.to_numpy(float)
phase0_ci=bootstrap_mean_ci(phase0_x,seed=SEED+41)
phase0_p=sign_flip_test(phase0_x,seed=SEED+41)
secondary_rows.append({"family":"secondary","endpoint":"phase0_primary_A_activity","n":50,
    "statistic":"two-sided model-level sign-flip","effect_size":float(phase0_x.mean()/phase0_x.std(ddof=1)),
    "estimate":float(phase0_x.mean()),"ci_low":float(phase0_ci[0]),"ci_high":float(phase0_ci[1]),
    "raw_p":float(phase0_p),"n_excluded":0,"note":"Exploratory phase-zero endpoint; not substituted for phase average."})
secondary_df=pd.DataFrame(secondary_rows)
secondary_df["holm_p"]=holm_adjust(secondary_df.raw_p.to_numpy(float))
secondary_df.to_csv(OUTPUT_DIR / "t4t5_secondary_stats.csv",index=False,float_format="%.17g")

subtype_rows=[]
for idx, subtype in enumerate(SUBTYPES):
    sg=activity_phase.loc[activity_phase.cell_group.eq(subtype)].groupby("model_id",as_index=False).A_activity.mean()
    sm=sg.merge(flow_model,on="model_id",validate="one_to_one")
    sx,sy=sm.A_activity.to_numpy(float),sm.A_flow.to_numpy(float)
    rho,prho=spearman_permutation(sx,sy,seed=SEED+100+idx)
    ci=bootstrap_spearman_ci(sx,sy,seed=SEED+200+idx)
    mci=bootstrap_mean_ci(sx,seed=SEED+300+idx)
    subtype_rows.append({"subtype":subtype,"n":50,"phase_mean_A_activity":float(sx.mean()),
        "mean_ci_low":float(mci[0]),"mean_ci_high":float(mci[1]),"spearman_rho_vs_A_flow":float(rho),
        "rho_ci_low":float(ci[0]),"rho_ci_high":float(ci[1]),"raw_p":float(prho),
        "note":"Exploratory; model-level units; all eight subtypes shown."})
subtype_df=pd.DataFrame(subtype_rows)
subtype_df["holm_p"]=holm_adjust(subtype_df.raw_p.to_numpy(float))
subtype_df.to_csv(OUTPUT_DIR / "t4t5_subtype_stats.csv",index=False,float_format="%.17g")

activity_phase.to_csv(OUTPUT_DIR / "t4t5_activity_phase_contrasts.csv",index=False,float_format="%.17g")
print("Secondary model-level analyses:")
print(secondary_df.to_string(index=False))
print("All subtype exploratory associations:")
print(subtype_df.to_string(index=False))
print("Primary model summaries:", len(model_summary), "models; exact-zero sign pairs excluded:", n_zero_excluded)

## 12. Secondary analyses (exploratory)

In [ ]:
assert secondary_df is not None and len(secondary_df) == 6
secondary_results = (
    activity_phase.groupby(["model_id","cell_group"],as_index=False)
    .agg(A_activity=("A_activity","mean"),
         phase_SD=("A_activity","std"),
         phase_sign_changes=("A_activity",lambda s: (lambda signs: bool(len(signs) > 1 and np.any(signs[1:] != signs[:-1])))(np.sign(s.to_numpy())[np.sign(s.to_numpy()) != 0])))
)
print("Secondary analyses summarized at the model level; p-values are a separate Holm family.")

## 13. Optional T4c phenotype linkage

In [ ]:
t4c_linkage = None
if A1_MEMBERSHIP.is_file():
    membership = pd.read_csv(A1_MEMBERSHIP)
    expected_membership_cols = {"model_id", "t4c_cluster_id", "t4c_phenotype"}
    if expected_membership_cols.issubset(membership.columns):
        assert len(membership) == 50 and membership.model_id.is_unique
        t4c_linkage = membership[list(expected_membership_cols)].copy()
        # This is an exploratory label join only; no mediation analysis is performed.
        print("A1 phenotype labels available for a later exploratory join:", len(t4c_linkage))
    else:
        print("A1 membership exists, but does not contain the expected phenotype label columns.")
else:
    print("A1 phenotype table is absent; primary B1 endpoints do not depend on it.")


## 14. Planned figures (disabled by default)

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import binomtest

assert model_summary.model_id.nunique() == 50
FIGURE_DIR = ROOT / "figures/supplementary/biological_constraints_B1"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# B1: model-level distribution with mean and bootstrap interval.
fig, ax = plt.subplots(figsize=(5.6,4.0))
vals=model_summary.phase_mean_A_activity.to_numpy(float)
parts=ax.violinplot([vals],positions=[0],showmedians=True,showextrema=False)
for body in parts["bodies"]:
    body.set_facecolor("#9ecae1"); body.set_edgecolor("#4C78A8"); body.set_alpha(.7)
rng=np.random.default_rng(SEED)
ax.scatter(rng.normal(0,.045,len(vals)),vals,s=24,alpha=.8,color="#315a7d",zorder=3)
ax.errorbar([0.20],[float(vals.mean())],
            yerr=[[float(vals.mean()-p1_ci[0])],[float(p1_ci[1]-vals.mean())]],
            fmt="o",color="#d62728",capsize=4,label="Mean and bootstrap 95% CI")
ax.axhline(0,color="0.4",lw=.8)
ax.set(xticks=[0],xticklabels=["50 models"],ylabel="Phase-averaged T4+T5 directional A")
ax.legend(frameon=False,fontsize=8)
fig.tight_layout(); fig.savefig(FIGURE_DIR/"B1_phase_averaged_A_activity.png",dpi=200); plt.close(fig)

# B2: paired model-level values; no fitted linear model.
fig,ax=plt.subplots(figsize=(5.3,4.4))
ax.scatter(model_summary.phase_mean_A_activity,model_summary.phase_mean_A_flow,
           s=30,alpha=.8,color="#4C78A8",edgecolor="white",linewidth=.3)
ax.axhline(0,color="0.65",lw=.8); ax.axvline(0,color="0.65",lw=.8)
ax.text(.03,.97,f"Spearman rho = {p2_rho:.3f}\n95% bootstrap CI [{p2_ci[0]:.3f}, {p2_ci[1]:.3f}]\npermutation p = {p2:.4g}",
        transform=ax.transAxes,ha="left",va="top",fontsize=8)
ax.set(xlabel="Phase-averaged T4+T5 directional A",ylabel="Phase-averaged decoded A_flow")
fig.tight_layout(); fig.savefig(FIGURE_DIR/"B2_A_activity_vs_A_flow.png",dpi=200); plt.close(fig)

# B3: complete 2 x 2 sign table after excluding exact zeros.
sign_table=np.zeros((2,2),dtype=int)
for a,f in zip(x[keep],y[keep]):
    sign_table[0 if a>0 else 1,0 if f>0 else 1]+=1
fig,ax=plt.subplots(figsize=(4.6,4.0))
im=ax.imshow(sign_table,cmap="Blues",vmin=0)
for i in range(2):
    for j in range(2):
        ax.text(j,i,str(sign_table[i,j]),ha="center",va="center",
                color="white" if sign_table[i,j]>sign_table.max()/2 else "black",fontsize=13)
ax.set(xticks=[0,1],xticklabels=["A_flow +","A_flow −"],
       yticks=[0,1],yticklabels=["A_activity +","A_activity −"],
       title=f"Sign agreement {n_agree}/{n_sign}; exact 95% CI [{p3_ci.low:.2f}, {p3_ci.high:.2f}]")
fig.colorbar(im,ax=ax,fraction=.046,pad=.04,label="Number of models")
fig.tight_layout(); fig.savefig(FIGURE_DIR/"B3_sign_agreement.png",dpi=200); plt.close(fig)

# B4: paired model-level phase means; secondary comparison.
fig,ax=plt.subplots(figsize=(5.3,4.2))
series=[
    model_summary.T4_phase_mean_A_activity.to_numpy(float),
    model_summary.T5_phase_mean_A_activity.to_numpy(float),
    model_summary.phase_mean_A_activity.to_numpy(float),
]
for i,row in enumerate(zip(*series)):
    ax.plot([0,1,2],row,color="0.72",alpha=.45,lw=.65,zorder=1)
for i,(arr,label,color) in enumerate(zip(series,["T4","T5","T4+T5"],["#4C78A8","#F58518","#54A24B"])):
    ax.scatter(np.full(len(arr),i),arr,s=16,alpha=.75,color=color,label=label,zorder=2)
    ax.scatter([i],[arr.mean()],marker="D",s=38,color="black",zorder=4)
ax.axhline(0,color="0.4",lw=.8)
ax.set(xticks=[0,1,2],xticklabels=["T4","T5","T4+T5"],
       ylabel="Phase-averaged directional A",title="Secondary: paired population readouts")
ax.legend(frameon=False,ncol=3,fontsize=8)
fig.tight_layout(); fig.savefig(FIGURE_DIR/"B4_T4_T5_comparison.png",dpi=200); plt.close(fig)

# B5: repeated phase trajectory; descriptive model bootstrap intervals, no phase-wise tests.
phase_summary=[]
for phase_index,g in primary_phase.groupby("phase_index",sort=True):
    vv=g.A_activity.to_numpy(float)
    ci=bootstrap_mean_ci(vv,seed=SEED+500+int(phase_index))
    phase_summary.append({"phase_index":int(phase_index),"phase_deg":float(g.phase_deg.iloc[0]),
                          "mean":float(vv.mean()),"ci_low":float(ci[0]),"ci_high":float(ci[1])})
phase_summary=pd.DataFrame(phase_summary)
fig,ax=plt.subplots(figsize=(6.0,4.0))
ax.plot(phase_summary.phase_deg,phase_summary["mean"],marker="o",color="#4C78A8")
ax.fill_between(phase_summary.phase_deg,phase_summary.ci_low,phase_summary.ci_high,
                color="#4C78A8",alpha=.2,label="Model bootstrap 95% CI")
ax.axhline(0,color="0.4",lw=.8)
ax.set(xlabel="Stimulus phase (degrees)",ylabel="Cohort mean T4+T5 directional A",
       title="Descriptive phase profile; phases are repeated within model")
ax.legend(frameon=False,fontsize=8)
fig.tight_layout(); fig.savefig(FIGURE_DIR/"B5_phase_profile_descriptive.png",dpi=200); plt.close(fig)
phase_summary.to_csv(OUTPUT_DIR/"t4t5_phase_descriptive.csv",index=False,float_format="%.17g")
print("Saved B1–B5 figures in",FIGURE_DIR)

## 15. QA and final decision template

In [ ]:
if primary_statistics is None or secondary_df is None:
    FINAL_VERDICT = "INCONCLUSIVE"
    print("B1 statistics are incomplete; no verdict can be made.")
else:
    primary_pass = bool((primary_statistics.holm_p.to_numpy(float) < 0.05).all())
    directional = bool(p2_rho > 0 and p3_prop > 0.5)
    FINAL_VERDICT = "YES" if primary_pass and directional else "INCONCLUSIVE"
    print({
        "N models": int(model_summary.model_id.nunique()),
        "N model-phase pairs": int(len(paired_phase)),
        "N activity rows": int(len(activity_long)),
        "P1 estimate/CI/raw p/Holm p": primary_statistics.iloc[0][["estimate","ci_low","ci_high","raw_p","holm_p"]].to_dict(),
        "P2 rho/CI/raw p/Holm p": primary_statistics.iloc[1][["estimate","ci_low","ci_high","raw_p","holm_p"]].to_dict(),
        "P3 agreement": f"{n_agree}/{n_sign}",
        "P3 proportion/CI/raw p/Holm p": primary_statistics.iloc[2][["estimate","ci_low","ci_high","raw_p","holm_p"]].to_dict(),
        "zero pairs excluded from P3": n_zero_excluded,
        "secondary T4/T5 means": {"T4":float(model_summary.T4_phase_mean_A_activity.mean()),
                                  "T5":float(model_summary.T5_phase_mean_A_activity.mean()),
                                  "T4+T5":float(model_summary.phase_mean_A_activity.mean())},
        "Holm-adjusted primary p-values": primary_statistics.holm_p.tolist(),
        "inference rerun": False,
        "verdict": FINAL_VERDICT,
        "interpretation": "NO is not assigned from nonsignificance; no equivalence margin was prespecified."
    })

## Interpretation and decision rules

- **YES** is reserved for all three prespecified primary endpoints passing Holm correction at 0.05: model-level mean A_activity, its model-level association with A_flow, and sign agreement.
- **NO** cannot be assigned from a nonsignificant test alone. A future NO verdict would require a justified equivalence bound specified before data review. Until then, nonsignificant results are **INCONCLUSIVE** and support only “the tested readout did not detect the signal.”
- Neither outcome establishes perception, causality, a biological mechanism, or decoder independence.
- Phases and cells are repeated/within-model measurements. The independent analysis unit is the trained network (N=50).
- The B1 readout is fixed before outcomes are computed; no sign reversal or weight fitting against decoded A_flow is permitted.
